# B3-J2 Atelier 2 — Cross-validation + 3 modèles + GridSearch

**Durée** : 1h30 (14h15-15h45)

## Objectifs

À la fin de cet atelier, vous savez :

1. Faire du **feature engineering** simple (créer 2 features dérivées).
2. Utiliser **`cross_val_score`** pour obtenir un score robuste (mean ± std).
3. Comparer **3 familles de modèles** (linéaire, ensemble bagging, ensemble boosting).
4. Optimiser des **hyperparamètres** avec **`GridSearchCV`**.
5. Évaluer **une seule fois** sur le test set et interpréter l'écart val/test.
6. **Bonus** : lire la **feature importance** d'un RandomForest.

**Dataset** : California Housing (suite logique de l'Atelier 1).

## Point de départ

On reprend le notebook de l'**Atelier 1** : baseline `LinearRegression` avec `R² ≈ 0.6` sur le set de validation. On a posé 4 questions à la fin :

1. Le score val est-il **stable** ? → **cross-validation**
2. Une LinearRegression capture-t-elle les **interactions** ? → on teste **RandomForest** et **GradientBoosting**
3. Quels **hyperparamètres** pour le meilleur modèle ? → **GridSearchCV**
4. Comment **comparer** rigoureusement ? → DataFrame récap mean/std

On y va.

In [ ]:
# Imports (étendus par rapport à l'atelier 1)
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    GridSearchCV,
    KFold,
)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

print("Imports OK")

### Recharger dataset + X/y/split (résumé court atelier 1)

In [ ]:
# Chargement
data = fetch_california_housing(as_frame=True)
df = data.frame.copy()

# Re-création de la feature catégorielle simulée (même seed que l'atelier 1)
np.random.seed(42)
df["OceanProximity"] = np.random.choice(
    ["NEAR_BAY", "INLAND", "NEAR_OCEAN", "ISLAND"],
    size=len(df),
    p=[0.25, 0.45, 0.25, 0.05],
)

print("Shape :", df.shape)
df.head(3)

## Étape 1/6 — Ajouter 2 features dérivées

**Feature engineering** : créer de nouvelles features à partir des existantes pour aider le modèle.

Idées dans le dataset California Housing :

- `RoomsPerHousehold` = `AveRooms` / `AveOccup` (plus de sens qu'AveRooms brut).
- `BedroomsRatio` = `AveBedrms` / `AveRooms` (proportion de chambres).
- `PopPerHousehold` = `Population` / `AveOccup`.

**À toi** : créer 2 features dérivées et les ajouter au DataFrame.

In [ ]:
# Feature 1 : ratio chambres / total pièces (proportion intuitive)
df["BedroomsRatio"] = df["AveBedrms"] / df["AveRooms"]

# Feature 2 : à compléter — ex : RoomsPerHousehold = AveRooms / AveOccup
df["RoomsPerHousehold"] = df["AveRooms"] / df["AveOccup"]

# Vérification : pas de NaN ni d'inf
new_features = ["BedroomsRatio", "RoomsPerHousehold"]
print(df[new_features].describe())
print("\nNaN ou inf :", df[new_features].replace([np.inf, -np.inf], np.nan).isna().sum().to_dict())

In [ ]:
# X / y
TARGET = "MedHouseVal"
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Split unique train/test (on fera la validation par cross-val intra-train)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42
)

print(f"Train : {X_train.shape}")
print(f"Test  : {X_test.shape}")

# Preprocessor (idem atelier 1)
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)
print("\nPreprocessor prêt.")

## Étape 2/6 — Cross-validation

**Problème** : un seul split train/val donne **un seul score**. Si tu changes `random_state`, le score change. Comment savoir si ton modèle est vraiment bon ?

**Solution** : **k-fold cross-validation**. On découpe le train en `k` plis :

```
fold 1 : [VAL] [train] [train] [train] [train]
fold 2 : [train] [VAL] [train] [train] [train]
fold 3 : [train] [train] [VAL] [train] [train]
fold 4 : [train] [train] [train] [VAL] [train]
fold 5 : [train] [train] [train] [train] [VAL]
```

On obtient **5 scores** → on rapporte **mean ± std**. Si `std` est grand, ton score est instable.

Note : `scoring="r2"` par défaut pour la régression. `n_jobs=-1` utilise tous les CPU.

In [ ]:
# Cross-val sur LinearRegression (baseline)
baseline_pipe = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", LinearRegression()),
    ]
)

# À compléter : appeler cross_val_score avec cv=5, scoring="r2", n_jobs=-1
scores = cross_val_score(baseline_pipe, X_train, y_train, cv=5, scoring="r2", n_jobs=-1)

print("Scores R² par fold :", np.round(scores, 3))
print(f"Mean ± std         : {scores.mean():.3f} ± {scores.std():.3f}")

**Lecture** : si la `std` est faible (< 0.02), le score est stable. Si elle est élevée, ton modèle est sensible au split = soit pas assez de données, soit modèle instable.

## Étape 3/6 — Tester 3 modèles

On compare 3 familles d'approches :

| Modèle | Famille | Force | Faiblesse |
|---|---|---|---|
| `LinearRegression` | Linéaire | Rapide, interprétable | Rate les interactions non-linéaires |
| `RandomForestRegressor` | Ensemble bagging | Capture interactions, robuste | Moins fin que boosting |
| `GradientBoostingRegressor` | Ensemble boosting | Souvent le meilleur sur tabulaire | Plus lent, plus sensible aux hyperparams |

In [ ]:
# Dictionnaire des modèles à comparer
models = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    # À compléter : ajouter GradientBoostingRegressor(random_state=42)
    "GradientBoosting": GradientBoostingRegressor(random_state=42),
}

results = []

for name, regressor in models.items():
    # À compléter : construire le pipeline preprocessor + regressor
    pipe = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("regressor", regressor),
        ]
    )

    # À compléter : mesurer le temps d'entraînement avec time.time()
    t0 = time.time()
    cv_scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring="r2", n_jobs=-1)
    elapsed = time.time() - t0

    results.append(
        {
            "model": name,
            "mean_cv_r2": cv_scores.mean(),
            "std_cv_r2": cv_scores.std(),
            "time_s": elapsed,
        }
    )
    print(f"{name:20s}  R² = {cv_scores.mean():.3f} ± {cv_scores.std():.3f}  ({elapsed:.1f}s)")

In [ ]:
# DataFrame récapitulatif trié par score
results_df = pd.DataFrame(results).sort_values("mean_cv_r2", ascending=False).reset_index(drop=True)
results_df.style.format({"mean_cv_r2": "{:.3f}", "std_cv_r2": "{:.3f}", "time_s": "{:.1f}"})

**Observation typique** : RandomForest et GradientBoosting battent LinearRegression de plusieurs points de R². L'écart entre les deux ensembles est plus mince.

**Choix** : on prend le meilleur des 3 pour la suite (probablement GradientBoosting ou RandomForest).

## Étape 4/6 — Hyperparamètres avec GridSearchCV

Chaque modèle a des **hyperparamètres** (paramètres que tu choisis, qui ne s'apprennent pas par `.fit()`) :

- `RandomForest` : `n_estimators` (nb d'arbres), `max_depth` (profondeur max).
- `GradientBoosting` : `n_estimators`, `learning_rate`, `max_depth`.

**GridSearchCV** essaie toutes les combinaisons d'une grille avec cross-validation, et retient la meilleure.

In [ ]:
# On lance GridSearch sur RandomForest (plus rapide à tuner que GB)
rf_pipe = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", RandomForestRegressor(random_state=42, n_jobs=-1)),
    ]
)

# À compléter : définir la grille de paramètres
# Note : on préfixe par "regressor__" car c'est le nom du step dans le Pipeline
param_grid = {
    "regressor__n_estimators": [50, 100, 200],
    "regressor__max_depth": [None, 10, 20],
}

grid_search = GridSearchCV(
    rf_pipe,
    param_grid=param_grid,
    cv=3,  # cv=3 pour gagner du temps en formation (cv=5 en prod)
    scoring="r2",
    n_jobs=-1,
    verbose=1,
)

t0 = time.time()
grid_search.fit(X_train, y_train)
print(f"\nGridSearch terminé en {time.time() - t0:.1f}s")

In [ ]:
# Afficher les meilleurs paramètres et le meilleur score
print("Best params  :", grid_search.best_params_)
print(f"Best CV R²   : {grid_search.best_score_:.3f}")

# Top 5 combinaisons
cv_results = pd.DataFrame(grid_search.cv_results_)
cols = ["param_regressor__n_estimators", "param_regressor__max_depth", "mean_test_score", "std_test_score"]
cv_results[cols].sort_values("mean_test_score", ascending=False).head()

## Étape 5/6 — Évaluation **une seule fois** sur test set

Règle d'or : le test set n'est ouvert qu'**à la fin**, sur **le modèle final**.

Si tu fais GridSearch puis tu testes 5 modèles sur le test pour choisir → tu fais du **leakage de test**.

Ici : on prend `grid_search.best_estimator_` (déjà refit sur tout le train) et on l'évalue **une fois** sur `X_test`.

In [ ]:
best_model = grid_search.best_estimator_

# À compléter : prédire sur X_test
y_pred_test = best_model.predict(X_test)

# À compléter : calculer MAE, RMSE, R² sur le test
mae_test = mean_absolute_error(y_test, y_pred_test)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
r2_test = r2_score(y_test, y_pred_test)

print("=== Scores TEST (one-shot) ===")
print(f"MAE test   : {mae_test:.3f}")
print(f"RMSE test  : {rmse_test:.3f}")
print(f"R²  test   : {r2_test:.3f}")
print(f"\nRappel CV R² (val) : {grid_search.best_score_:.3f}")
print(f"Écart val → test   : {r2_test - grid_search.best_score_:+.3f}")

## Étape 6/6 BONUS — Feature importance

Un `RandomForest` (et `GradientBoosting`) expose `.feature_importances_` : pour chaque feature, l'importance moyenne dans les arbres.

**Utilité** :
- Comprendre **ce qui drive** la prédiction.
- Détecter une feature suspecte (importance trop haute → leakage probable).
- Communiquer à un non-data avec un graphique simple.

In [ ]:
# Récupérer le RandomForest fitté dans le pipeline
rf = best_model.named_steps["regressor"]

# Récupérer les noms des features après preprocessing (OneHot inclus)
preproc_fitted = best_model.named_steps["preprocessor"]
feature_names = preproc_fitted.get_feature_names_out()

# À compléter : récupérer rf.feature_importances_
importances = rf.feature_importances_

fi_df = (
    pd.DataFrame({"feature": feature_names, "importance": importances})
    .sort_values("importance", ascending=True)
    .tail(15)  # top 15
)

plt.figure(figsize=(8, 6))
plt.barh(fi_df["feature"], fi_df["importance"])
plt.xlabel("Importance")
plt.title("Top 15 — Feature importance (RandomForest)")
plt.tight_layout()
plt.show()

fi_df.iloc[::-1].head(10)

## Discussion en groupe

**Question 1 — Pourquoi test < val parfois ? C'est mauvais signe ?**

Plusieurs causes possibles :

1. **Variance naturelle** : avec un test de 15%, ±0.01-0.02 de R² est normal. Pas dramatique.
2. **Sur-optim sur val** : si tu as fait beaucoup d'itérations de GridSearch, ton modèle s'est ajusté aux particularités du fold val.
3. **Distribution shift** : si train et test ne sont pas tirés de la même distribution (ex : données temporelles). Pas le cas ici mais c'est **le** piège de cet après-midi (cas hippique).
4. **Test set trop petit** : pas assez statistiquement représentatif.

**Question 2 — Et si test > val ?**

C'est suspect aussi. Ça arrive (chance du split), mais ça peut signaler que ton val était particulièrement difficile, ou que tu as sous-estimé ton modèle. Ne pas se réjouir trop vite.

## Récap acquis — Atelier 2

- **Feature engineering** simple (2 ratios dérivés) qui peut suffire à gagner quelques points de R².
- **Cross-validation 5-fold** = score robuste avec écart-type, plutôt qu'un score unique fragile.
- **Comparaison de 3 modèles** dans un DataFrame récap (mean / std / temps).
- **GridSearchCV** pour tuner les hyperparamètres du meilleur candidat.
- **Évaluation test one-shot** — règle d'or : le test n'est pas un outil de sélection de modèle.
- **Feature importance** pour comprendre ce que le modèle a appris.

## Transition — cas hippique (16h)

On a fait tout ça sur des données **IID** (indépendantes, identiquement distribuées). Random split → pas de problème.

**L'après-midi on voit pourquoi sur données temporelles (courses hippiques) tout change** :

- Random split = **data leakage garanti** (tu utilises le futur pour prédire le passé).
- Il faut **temporal CV** : train mois 1-6, test mois 7.
- Et même là, le backtest n'est **pas** un audit de production.